# Análisis Exploratorio de Datos (EDA)

In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
df_train = pd.read_csv('../data/splits/train_before_eda.csv')
df_train.head()
df_train.tail()

In [ ]:
print(f"Dimensiones del dataset de entrenamiento: {df_train.shape}")   # (registros, variables)
df_train.info(memory_usage="deep")
resumen = pd.DataFrame({
    "tipo": df_train.dtypes,
    "n_unicos": df_train.nunique(),
    "n_faltantes": df_train.isna().sum(),
    "%_faltantes": (df_train.isna().mean() * 100).round(2),
})
resumen

## 1. Entender el Target
Nuestra variable objetivo es `is_canceled`, donde:
* `1` = La reserva fue cancelada.
* `0` = La reserva no fue cancelada (se concretó o está activa).
El objetivo de negocio es predecir si una reserva será cancelada para poder tomar medidas preventivas.

## 2. Revisar estructura de datos
Vamos a ver un resumen estadístico de las variables numéricas y categóricas.

In [ ]:
# Resumen numérico
display(df_train.describe())

# Resumen categórico
display(df_train.describe(include=['object', 'category']))

## 3. Evaluar missing y calidad
Analizamos las variables con valores nulos y chequeamos posibles inconsistencias (ej. reservas sin huéspedes).

In [ ]:
# Variables con nulos
nulos = df_train.isna().sum()
print("Variables con nulos:")
print(nulos[nulos > 0])

# Inconsistencias: reservas sin huéspedes
sin_huespedes = df_train[(df_train['adults'] == 0) & (df_train['children'] == 0) & (df_train['babies'] == 0)]
print(f"\nReservas sin huéspedes: {len(sin_huespedes)}")

# Inconsistencias: ADR negativo
adr_negativo = df_train[df_train['adr'] < 0]
print(f"Reservas con ADR negativo: {len(adr_negativo)}")

## 4. Analizar target
Veamos la distribución de nuestra variable objetivo para entender si hay desbalance de clases.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

sns.countplot(data=df_train, x='is_canceled')
plt.title('Distribución del Target (is_canceled)')
plt.show()

print("Proporción:")
print(df_train['is_canceled'].value_counts(normalize=True))

## 5. Analizar predictores individualmente
Visualizamos algunas variables clave.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=df_train, x='lead_time', bins=50, ax=axes[0])
axes[0].set_title('Distribución de lead_time')

sns.countplot(data=df_train, x='hotel', ax=axes[1])
axes[1].set_title('Distribución por tipo de hotel')
plt.show()

## 6. Relacionar cada predictor con el target
¿Cómo afectan distintas variables a la tasa de cancelación?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=df_train, x='hotel', y='is_canceled', ax=axes[0])
axes[0].set_title('Tasa de cancelación por tipo de hotel')

sns.boxplot(data=df_train, x='is_canceled', y='lead_time', ax=axes[1])
axes[1].set_title('Lead time vs Cancelación')
plt.show()

## 7. Estudiar correlaciones entre predictores
Analizamos la colinealidad entre variables numéricas.

In [ ]:
num_cols = df_train.select_dtypes(include=['int64', 'float64']).columns
corr = df_train[num_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
plt.title('Matriz de Correlación')
plt.show()

## 8. Detectar outliers
Buscamos valores atípicos, especialmente en `adr`.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_train, x='adr')
plt.title('Boxplot de ADR')
plt.show()

## 9. Evaluar transformaciones
Algunas variables como `lead_time` tienen un sesgo fuerte a la derecha.

In [ ]:
print(f"Skewness de lead_time: {df_train['lead_time'].skew()}")

## 10. Detectar leakage (Fuga de información)

Nuestro objetivo es predecir la variable `is_canceled`. Dentro del dataset contamos con las siguientes variables:
* `reservation_status`: último estado de la reserva (Canceled, Check-Out, No-Show)
* `reservation_status_date`: fecha en la que se estableció dicho estado.

Si bien ambas variables están estrechamente relacionadas con la variable objetivo, presentan un problema desde el punto de vista predictivo: solo se conocen una vez que la reserva ya alcanzó su desenlace o estado final. Por lo tanto, utilizar esta información para predecir `is_canceled` introduciría data leakage, ya que estaríamos incorporando información que no estaría disponible al momento de realizar la predicción.

De todos modos, considero que sería útil analizar la relación entre `is_canceled` y `reservation_status` como parte de la etapa de validación y comprensión de los datos. En principio, esperaríamos que los registros con `is_canceled = 1` coincidieran con `reservation_status = "Canceled"`. Verificar esta correspondencia nos permitiría confirmar la consistencia de la información y detectar posibles discrepancias.

Este análisis sirve como fundamento metodológico para justificar la exclusión de estas variables del modelo predictivo. Además, nos permite documentar que la decisión no responde únicamente a una cuestión conceptual, sino también a la naturaleza temporal de los datos.

In [ ]:
# Análisis de consistencia
crosstab = pd.crosstab(df_train['is_canceled'], df_train['reservation_status'])
display(crosstab)

## 11. Seleccionar variables candidatas y 12. Documentar hallazgos

**Hallazgos principales:**
1. **Calidad de datos:** Hay reservas sin huéspedes que deberían ser eliminadas. Hay valores nulos en `company`, `agent`, `country` y `children` que requerirán imputación o eliminación.
2. **Outliers:** Existen valores atípicos extremos en `adr` que deben ser tratados.
3. **Leakage:** Las variables `reservation_status` y `reservation_status_date` deben ser eliminadas antes del modelado para evitar data leakage, tal como validamos en el paso anterior.
4. **Transformaciones:** Variables sesgadas como `lead_time` podrían beneficiarse de transformaciones logarítmicas.

Estos hallazgos guiarán el notebook de preprocesamiento.